In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

In [2]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### Defaults

In [3]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)


['subtype_for_PS_TCGA-ACC.tsv',
 'subtype_for_PS_TCGA-HNSC.tsv',
 'subtype_for_PS_TCGA-KIRP.tsv',
 'subtype_for_PS_TCGA-OV.tsv',
 'stage_for_PS_TCGA-UCEC_Subtype_basaloid squamous cell carcinoma.tsv',
 'subtype_for_PS_TCGA-UCS.tsv',
 'subtype_for_PS_TCGA-PAAD.tsv',
 'stage_for_PS_TCGA-UCEC_Subtype_papillary serous cystadenocarcinoma.tsv',
 'subtype_for_PS_TCGA-LIHC.tsv',
 'subtype_for_PS_TCGA-STAD.tsv',
 'subtype_for_PS_TCGA-KIRC.tsv',
 'subtype_for_PS_TCGA-PRAD.tsv',
 'subtype_for_PS_TCGA-BLCA.tsv',
 'subtype_for_PS_TCGA-THYM.tsv',
 'subtype_for_PS_TCGA-COAD.tsv',
 'subtype_for_PS_TCGA-GBM.tsv',
 'subtype_for_PS_TCGA-PCPG.tsv',
 'programs.txt',
 'subtype_for_PS_TCGA-SKCM.tsv',
 'subtype_for_PS_TCGA-KICH.tsv',
 'subtype_for_PS_TCGA-CESC.tsv',
 'primary_site_program_TCGA.tsv',
 'subtype_for_PS_TCGA-TGCT.tsv',
 'subtype_for_PS_TCGA-THCA.tsv',
 'subtype_for_PS_TCGA-UVM.tsv',
 'subtype_for_PS_TCGA-LAML.tsv',
 'subtype_for_PS_TCGA-DLBC.tsv',
 'subtype_for_PS_TCGA-READ.tsv',
 'subtype_for_PS

### Get GDC programs - get_gdc_progams()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "facets": "program.name"  

### Get valid primary sites - get_primary_sites()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "fields": "project_id,name,primary_site,disease_type"  

### Get valid subtypes - get_subtypes()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> facets = "diagnoses.primary_diagnosis"  

### For each subtype → get stages  - get_stages()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> "field": "diagnoses.ajcc_pathologic_stage" - AJCC diagnoses   

### For each (subtype, stage) → get_samples()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> given pid, subtype, stage  
> from cases   
> "samples.sample_id", "samples.submitter_id", "samples.sample_type"  

### get RNA-seq files - get_expression_files_given_samples()

> given: "field": "cases.case_id", "value": case_ids  
> end_point: url_gdc_files = "https://api.gdc.cancer.gov/files"  



### All programs

In [4]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


In [5]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a programa

In [6]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [7]:
force=False
verbose=True

program = 'TCGA'

dfc = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

print(len(dfc))
dfc.head(7)


Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/tcga/primary_site_program_TCGA.tsv'
33


,pid,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma
3,TCGA-LGG,Brain,TCGA-LGG,Gliomas,Brain Lower Grade Glioma
4,TCGA-GBM,Brain,TCGA-GBM,"Not Reported, Gliomas",Glioblastoma Multiforme
5,TCGA-BRCA,Breast,TCGA-BRCA,"Adnexal and Skin Appendage Neoplasms, Adenomas...",Breast Invasive Carcinoma
6,TCGA-LUAD,Bronchus and lung,TCGA-LUAD,"Ductal and Lobular Neoplasms, Cystic, Mucinous...",Lung Adenocarcinoma


In [8]:
dfc.tail(3)

,pid,primary_site,project_id,disease_type,name
30,TCGA-THCA,Thyroid gland,TCGA-THCA,"Squamous Cell Neoplasms, Epithelial Neoplasms,...",Thyroid Carcinoma
31,TCGA-UCS,"Uterus, NOS, Corpus uteri",TCGA-UCS,"Basal Cell Neoplasms, Complex Mixed and Stroma...",Uterine Carcinosarcoma
32,TCGA-UCEC,"Uterus, NOS, Corpus uteri",TCGA-UCEC,"Not Reported, Cystic, Mucinous and Serous Neop...",Uterine Corpus Endometrial Carcinoma


### Subtypes

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

In [9]:
force=True
verbose=True

i = 0
pid = dfc.iloc[i].pid
primary_site = dfc.iloc[i].primary_site

print(i, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

print("df_cases", len(df_cases), "df_subt", len(df_subt))
df_subt

0 TCGA-ACC Adrenal gland
Searching: ..

👉 Returned 92 / Total paginated 92
> 1
----------- 1 ---------------
rows 92 
columns Index(['id', 'primary_site', 'disease_type', 'case_id', 'diagnoses',
       'project.project_id'],
      dtype='object')
---------------------------
>>> main_diag Adrenal cortical carcinoma ajcc_clinical_stage
>>> list [{'primary_diagnosis': 'Adrenal cortical carcinoma', 'ajcc_pathologic_stage': None, 'tumor_grade': None}]
>>> found1111? {'primary_diagnosis': 'Adrenal cortical carcinoma', 'ajcc_pathologic_stage': None, 'tumor_grade': None}
>>> main_diag Adrenal cortical carcinoma ajcc_clinical_stage
>>> list [{'primary_diagnosis': 'Adrenal cortical carcinoma', 'ajcc_pathologic_stage': None, 'tumor_grade': None}, {'primary_diagnosis': 'Adrenal cortical carcinoma'}]
>>> found1111? {'primary_diagnosis': 'Adrenal cortical carcinoma', 'ajcc_pathologic_stage': None, 'tumor_grade': None}
>>> found1111? {'primary_diagnosis': 'Adrenal cortical carcinoma'}
>>> main_diag A

""


In [ ]:
df_cases.head(3).T

In [ ]:
df_cases.stage

In [ ]:
df_cases.pid.unique()

In [ ]:
for i in range(len(df_cases)):
    print(i, " - ", df_cases.diagnoses.iloc[i])

In [ ]:
df_subt

In [13]:

def extract_any(x, key, debug:bool=False):
    if debug: print("#", x)

    if isinstance(x, list):
        for d in x:
            if isinstance(d, dict) and key in d and d[key] is not None:
                return d[key]
    elif isinstance(x, dict):
        return x.get(key)
    return None

def extract_all(x, key, main_diag):
    print(">>> main_diag", main_diag, key)
    dicf = {"diagnosis": 'unknown',
            "ajcc": 'unknown'
            }
    
    if isinstance(x, list):
        print(">>> list", x)
        for d in x:
            if isinstance(d, dict):
                print(">>> found?", d)
                diag_aux = d.get("primary_diagnosis")

                if diag_aux == main_diag:
                    dicf = {"diagnosis": main_diag,
                            "ajcc": 'unknown'
                            }
                    if key in d.keys():
                        print(">>> found222?", d)
                        dicf = {"diagnosis": main_diag,
                                "ajcc": d.get(key)
                                }
                        break

    elif isinstance(x, dict):
        print(">>> dict", x)
        dicf = {"diagnosis": x.get("primary_diagnosis"),
                "ajcc": x.get(key)
                }

    return dicf

diag_list = df_cases["diagnoses"].map(lambda x: extract_any(x, "primary_diagnosis"))
diag_list = [x for x in diag_list if isinstance(x, str) and x.strip()]
dic = Counter(diag_list)

dfa = pd.DataFrame({
    'diag': list(dic.keys()),
    'n': list(dic.values())
})

dfa = dfa.sort_values('n', ascending=False)
main_diag = dfa.iloc[0].diag


dicf = df_cases["diagnoses"].map(lambda x: extract_all(x, "ajcc_pathologic_stage", main_diag))
dicf

>>> main_diag Adrenal cortical carcinoma ajcc_pathologic_stage
>>> list [{'primary_diagnosis': 'Adrenal cortical carcinoma', 'ajcc_pathologic_stage': None, 'tumor_grade': None}]
>>> found? {'primary_diagnosis': 'Adrenal cortical carcinoma', 'ajcc_pathologic_stage': None, 'tumor_grade': None}
>>> found222? {'primary_diagnosis': 'Adrenal cortical carcinoma', 'ajcc_pathologic_stage': None, 'tumor_grade': None}
>>> main_diag Adrenal cortical carcinoma ajcc_pathologic_stage
>>> list [{'primary_diagnosis': 'Adrenal cortical carcinoma', 'ajcc_pathologic_stage': None, 'tumor_grade': None}, {'primary_diagnosis': 'Adrenal cortical carcinoma'}]
>>> found? {'primary_diagnosis': 'Adrenal cortical carcinoma', 'ajcc_pathologic_stage': None, 'tumor_grade': None}
>>> found222? {'primary_diagnosis': 'Adrenal cortical carcinoma', 'ajcc_pathologic_stage': None, 'tumor_grade': None}
>>> main_diag Adrenal cortical carcinoma ajcc_pathologic_stage
>>> list [{'primary_diagnosis': 'Adrenal cortical carcinoma', 

0     {'diagnosis': 'Adrenal cortical carcinoma', 'a...
1     {'diagnosis': 'Adrenal cortical carcinoma', 'a...
2     {'diagnosis': 'Adrenal cortical carcinoma', 'a...
3     {'diagnosis': 'Adrenal cortical carcinoma', 'a...
4     {'diagnosis': 'Adrenal cortical carcinoma', 'a...
                            ...                        
87    {'diagnosis': 'Adrenal cortical carcinoma', 'a...
88    {'diagnosis': 'Adrenal cortical carcinoma', 'a...
89    {'diagnosis': 'Adrenal cortical carcinoma', 'a...
90    {'diagnosis': 'Adrenal cortical carcinoma', 'a...
91    {'diagnosis': 'Adrenal cortical carcinoma', 'a...
Name: diagnoses, Length: 92, dtype: object

In [ ]:
diag_list = df_cases["diagnoses"].map(lambda x: extract_any(x, "primary_diagnosis"))
dic = Counter(diag_list)

dic = dict(dic)
diags = list(dic.keys())
ns = list(dic.values())

dfa = pd.DataFrame({'diag': diags, 'n': ns})
dfa = dfa.dropna(subset=["diag"])
dfa = dfa.sort_values('n', ascending=False)

main_diag = dfa.iloc[0].diag.lower().strip()
print(main_diag)
dfa

In [ ]:
df_prof

In [ ]:
df_cases[df_cases["tumor_class"] == "other"]["primary_diagnosis"].value_counts()

In [ ]:
force=True
verbose=True
run_all = False

if run_all:
    df_subt = pd.DataFrame()

    for i in range(len(dfc)):
        print(i, end=' ')
        pid = dfc.iloc[i].pid
        primary_site = dfc.iloc[i].primary_site

        print(pid, primary_site)

        df_subt = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=True, force=force, verbose=verbose)
else:
    i = 31
    pid = dfc.iloc[i].pid
    primary_site = dfc.iloc[i].primary_site

    print(i, pid, primary_site)

    df_subt = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=True, force=force, verbose=verbose)
    

print(len(df_subt))
df_subt

### Stages

In [ ]:
force=True
verbose=True

i=1
subtype = df_subt.iloc[i].subtype_raw

print(f"pid '{pid}', subtype '{subtype}'")

df_stage = gdc.get_stages(pid=pid, subtype=subtype, do_filter=True, force=force, verbose=verbose)

print(len(df_stage))
df_stage

In [ ]:
force=False
verbose=True

i=0
stage = df_stage.iloc[i].stage_raw


print(pid, subtype, stage)

df_case = gdc.get_cases(pid=pid, subtype=subtype, stage=stage, force=force, verbose=verbose)
print(len(df_case))
df_case.head(12)

### Given a PS, Subtype, and Stage --> return the SAMPLES

In [ ]:
force=False
verbose=True

print(pid, subtype, stage)

df_sample = gdc.get_samples(pid=pid, subtype=subtype, stage=stage, batch_size=200, force=force, verbose=verbose)
print(len(df_sample))
df_sample.head(8)

In [ ]:
force=False
verbose=True

case_ids = list(np.unique(df_sample.case_id))

print(pid, subtype, stage, f"cases {len(case_ids)}")

df_files = gdc.get_expression_files_given_samples(pid=pid, subtype=subtype, stage=stage, 
                                                  case_ids=case_ids, batch_size=20, force=force, verbose=verbose)
print(len(df_files))
df_files.head(6)

In [ ]:
case_ids[:10]